# Convert the Adelani English→Yoruba model for in-browser use

Converts [`Davlan/m2m100_418M-eng-yor-mt`](https://huggingface.co/Davlan/m2m100_418M-eng-yor-mt) (David Adelani, Masakhane) to ONNX so the Consent-First Kit can run it **in the programme lead's browser** with Transformers.js: free, no server, text never leaves the device.

Run on free Google Colab (CPU is enough, about 10 minutes). Run cells top to bottom.

**Before publishing:** the original model page states no licence. Credit is included below; confirm reuse terms with the author before wide use.

## 1. Install the official Transformers.js conversion tooling

In [ ]:
%cd /content
!rm -rf transformers.js
!git clone --depth 1 --branch 3.8.1 https://github.com/huggingface/transformers.js.git
%cd /content/transformers.js
# Colab's Python has no ready-made wheel for the pinned onnx 1.17; 1.18 is compatible
!sed -i -E 's/^onnx==.*/onnx==1.18.0/' scripts/requirements.txt
!pip install -q -r scripts/requirements.txt
# Colab extras that break the exporter: diffusers needs a newer huggingface_hub, and torch >= 2.9 needs onnxscript.
# torch 2.7 uses the classic, well-tested ONNX exporter (the conversion runs in its own process, so no restart needed).
!pip uninstall -y -q diffusers torchvision torchaudio  # torchvision/torchaudio must match torch; not needed here
!pip install -q torch==2.7.1 onnxscript

## 2. Export to ONNX
Exported without cached decoding (`text2text-generation`) and without post-processing: the full export runs out of memory on free Colab. Short form lines translate fine this way.

In [ ]:
%cd /content
!optimum-cli export onnx --model Davlan/m2m100_418M-eng-yor-mt --task text2text-generation --no-post-process m2m_onnx
!ls -la m2m_onnx

## 3. Compress to 8-bit and check translations before publishing
Restart the session first (Runtime > Restart session) so the newly installed torch is used. Files on disk are kept.

In [ ]:
import os, shutil
from onnxruntime.quantization import quantize_dynamic, QuantType
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer

os.makedirs('m2m_onnx/onnx', exist_ok=True)
for n in ['encoder_model', 'decoder_model']:
    quantize_dynamic(f'm2m_onnx/{n}.onnx', f'm2m_onnx/onnx/{n}_quantized.onnx', weight_type=QuantType.QInt8)
    shutil.move(f'm2m_onnx/{n}.onnx', f'm2m_onnx/onnx/{n}.onnx')

tok = AutoTokenizer.from_pretrained('m2m_onnx')
tok.src_lang = 'en'
mdl = ORTModelForSeq2SeqLM.from_pretrained('m2m_onnx', subfolder='onnx', encoder_file_name='encoder_model_quantized.onnx',
                                           decoder_file_name='decoder_model_quantized.onnx', use_cache=False)
for line in ['We would like to ask you a few questions about how you have been feeling.',
             'You can say no to any part, and you can change your mind later.',
             'Badly most nights']:
    ids = tok(line, return_tensors='pt')
    out = mdl.generate(**ids, forced_bos_token_id=tok.get_lang_id('yo'), max_new_tokens=100)
    print(line, ' -> ', tok.batch_decode(out, skip_special_tokens=True)[0])
!ls -la m2m_onnx/onnx

## 4. Sign in to Hugging Face
Save a **write** token in Colab Secrets (key icon, left) under the name `Hugging_face_key` and switch on notebook access. The cell reads it by name; never paste the token into a cell.

In [ ]:
# Uses the token saved in Colab Secrets (key icon) under the NAME Hugging_face_key.
# Never type the token itself into a cell.
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('Hugging_face_key'))

## 5. Publish the browser-ready copy with credit to the original author

In [ ]:
import shutil
from huggingface_hub import HfApi, whoami
REPO = f"{whoami()['name']}/m2m100_418M-eng-yor-mt-onnx"
readme = f'''---
base_model: Davlan/m2m100_418M-eng-yor-mt
library_name: transformers.js
pipeline_tag: translation
language: [en, yo]
---
# m2m100_418M-eng-yor-mt (ONNX, for Transformers.js)

Browser-ready conversion of [Davlan/m2m100_418M-eng-yor-mt](https://huggingface.co/Davlan/m2m100_418M-eng-yor-mt)
by David Adelani (Masakhane), fine-tuned on JW300 and MENYO-20k. All credit to the original authors; no weights were changed.

Used by the [Consent-First Kit](https://github.com/SDBE-Ex/consent-first-kit) to give programme leads a second,
locally made Yoruba **draft** that a native speaker reviews before anything reaches people.

Licence: the original model card states no licence. Reuse terms are being confirmed with the author.
'''
open('m2m_onnx/README.md', 'w').write(readme)
# Transformers.js looks for decoder_model_merged; this decoder has no cache inputs, which it handles.
shutil.copy('m2m_onnx/onnx/decoder_model_quantized.onnx', 'm2m_onnx/onnx/decoder_model_merged_quantized.onnx')
api = HfApi()
api.create_repo(REPO, exist_ok=True)
# Upload only the compressed model plus tokenizer files (the full-size copies are ~3 GB)
api.upload_folder(folder_path='m2m_onnx', repo_id=REPO, allow_patterns=['*.json', '*.model', 'README.md', 'onnx/*_quantized.onnx'])
print('Published:', f'https://huggingface.co/{REPO}')
print('Set this in Vercel as VITE_LOCAL_YO_MODEL =', REPO)

## 6. Add the fast tokenizer file Transformers.js needs
The original repo has only the slow SentencePiece tokenizer, and Transformers.js needs `tokenizer.json`. The vocabulary is identical to `Xenova/m2m100_418M`, so we check that and copy its `tokenizer.json`.

Note for the app: this copy has **no cached decoding**, so the app's worker runs its own full-sequence greedy loop instead of Transformers.js `generate()` (which would return an empty draft).

In [ ]:
from huggingface_hub import hf_hub_download, HfApi, whoami
import json
REPO = f"{whoami()['name']}/m2m100_418M-eng-yor-mt-onnx"
ours = json.load(open(hf_hub_download('Davlan/m2m100_418M-eng-yor-mt', 'vocab.json')))
tj = hf_hub_download('Xenova/m2m100_418M', 'tokenizer.json')
theirs = json.load(open(tj))['model']['vocab']
print('same vocab:', ours == theirs)
assert ours == theirs, 'vocab differs: do not copy'
HfApi().upload_file(path_or_fileobj=tj, path_in_repo='tokenizer.json', repo_id=REPO)
print('Uploaded tokenizer.json to', REPO)